# Histology-guided spatial deconvolution

Fit RNA and registered H&E morphology with shared spatial component weights.
Required: `counts.h5ad` with raw counts and graph coordinates in `obsm["spatial"]`.
Provide either a precomputed HIPT/UNI morphology matrix in `obsm["morphology"]`, or an H&E
image plus `image_spots.csv` (spot IDs, `x,y` centers in original image pixels), image resolution
and capture radius. Optional region labels may be stored in `adata.obs`.

Both model arms start from the same RNA consensus. The same morphology matrix supplies image
calibration, reconstruction and morphology-dependent edge weights; the RNA-only arm reuses
the calibrated coordinate-TV coefficient. Both terms retain the fitting exposure weights.

## Data links

- HER2ST G2/H1 counts, H&E images, coordinates and region annotations: [Zenodo 4751624](https://doi.org/10.5281/zenodo.4751624).

## Example figures

Archived G2 example. Columns, left to right: registered H&E, pathology annotations,
RNA-only fit and RNA + H&E fit. The lower row enlarges the boxed region. The following
plots compare region-wise correlations and overall scores on the same annotated spots.

![G2 H&E, pathology, RNA-only and RNA plus H&E maps with matching enlargements](../docs/source/_static/figures/g2_regions.png)

![G2 region-wise correlations for the paired model arms](../docs/source/_static/figures/g2_region_pcc.png)

![G2 ARI, mean correlation and boundary F1 for the paired model arms](../docs/source/_static/figures/g2_improvement.png)


## 1. Configure your input

Install the notebook dependencies, then edit `DATA_DIR`, the input filenames and `K` below.
Supply your own `.h5ad` files with unique spot and feature IDs. RNA fitting uses raw counts;
set `COUNTS_LAYER="counts"` when they are stored in a layer. Protein inputs should contain
your prepared nonnegative assay measurements. Each fitted spot needs positive total signal.

Running the fitting cells trains a new model. `DEVICE` accepts `"auto"`, `"cpu"` or `"cuda"`.
The study's reference-free and joint analyses used GPUs. Inspect the returned diagnostics
to determine whether your fit converged. Large inputs require enough memory for dense matrices.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from IPython.display import display
from tutorials.common import correlation, align, spatial, pies, legend, save_figure, save_table
from spDecon.reproducibility import set_seed

DATA_DIR = Path("/path/to/your/data")
COUNTS_LAYER = None
DEVICE = "auto"
VERBOSE = True
MAX_ITER = 20000
set_seed(1, deterministic=True)

K = 6  # Choose the component number for your data.
EXTRACT_IMAGE = False
REGION_COLUMN = None  # Optional column name in adata.obs
MICRONS_PER_PIXEL = None  # Set when extracting image features
RADIUS_PIXELS = None      # Capture radius in original image pixels


## 2. Read molecular measurements


In [ ]:
adata = ad.read_h5ad(DATA_DIR / "counts.h5ad")
y = np.array(adata.to_df(layer=COUNTS_LAYER), dtype=float, order="F")
xy = np.asarray(adata.obsm["spatial"])
sections = adata.obs["section_id"].to_numpy() if "section_id" in adata.obs else np.repeat("section_1", len(y))
print(adata)


## 3. Extract HIPT and UNI feature embeddings

The feature-extraction code is included in spDecon. From the repository root, install the optional image dependencies and prepare a directory for HIPT weights:

```bash
python -m pip install -r requirements-histology.lock.txt
# CPU alternative: see the installation guide for CPU-only torchvision.
mkdir -p checkpoints/hipt
hf auth login
```

For a CPU-only environment, follow the [CPU image-dependency instructions](https://dsong-lab.github.io/spDecon/installation.html#optional-h-e-feature-extraction) instead of installing the CUDA lock file.

Download `vit256_small_dino.pth` and `vit4k_xs_dino.pth` from the [official HIPT repository](https://github.com/mahmoodlab/HIPT) into `checkpoints/hipt/`. Request access to [MahmoodLab/UNI](https://huggingface.co/MahmoodLab/UNI) using the same account as `hf auth login`. Weights are obtained under their original terms; no token needs to be written in this notebook.

The following code runs the complete image pipeline:

1. Rescale the registered RGB H&E image, spot centers and capture radius to 0.5 μm/pixel.
2. Extract HIPT and UNI embedding maps. For each encoder, apply PCA retaining 99% of variance and fit a 16-word visual dictionary.
3. Pool dictionary-word frequencies over each spot's capture footprint. Concatenate HIPT and UNI with equal half-mass to obtain the nonnegative, 32-column morphology matrix `q`.

`extract_histology` returns this spot-indexed matrix and saves intermediate embeddings as `embeddings-hist-vit.pickle` and `embeddings-hist-uni.pickle`. The output directory also contains `hipt_pca99.npy`, `uni_pca99.npy`, PCA transforms, dictionaries, and the final `morphology.npz` and `morphology.csv`. The model receives the pooled frequencies in `q`; intermediate embeddings and PCA scores can contain signed values.

Set `EXTRACT_IMAGE=True` when starting from an image. Otherwise supply your previously extracted matrix in `adata.obsm["morphology"]`. RNA measurements and pathology labels do not enter image-feature construction.


In [ ]:
from spDecon.image_features import extract_histology
if EXTRACT_IMAGE:
    if MICRONS_PER_PIXEL is None or RADIUS_PIXELS is None:
        raise ValueError("Set MICRONS_PER_PIXEL and RADIUS_PIXELS for your image")
    image_spots = pd.read_csv(DATA_DIR / "image_spots.csv", index_col=0).loc[adata.obs_names]
    q_frame = extract_histology(
        DATA_DIR / "he.jpg", image_spots,
        microns_per_pixel=MICRONS_PER_PIXEL, radius_pixels=RADIUS_PIXELS,
        checkpoint_dir=ROOT / "checkpoints/hipt", output=ROOT / "outputs/histology_features",
        device=DEVICE)
    adata.obsm["morphology"] = q_frame.loc[adata.obs_names].to_numpy()
if "morphology" not in adata.obsm:
    raise ValueError("Supply adata.obsm['morphology'] or enable EXTRACT_IMAGE")
q = np.asarray(adata.obsm["morphology"], dtype=float)
if q.shape != (len(y), 32) or not np.isfinite(q).all() or (q < 0).any():
    raise ValueError("Morphology must be a finite, nonnegative spots-by-32 matrix")
np.testing.assert_allclose(q.reshape(len(q), 2, 16).sum(axis=2), .5, atol=1e-8, rtol=0)


## 4. Calibrate and fit paired models

Image calibration uses multiplier 0.30; spatial calibration uses 0.05 on the coordinate graph
before morphology gating. Coefficients are recalculated for your input. Precomputed features can
be reloaded from the extractor's `morphology.npz` using its `Q` and `spots` arrays; align their rows
to `adata.obs_names` before fitting. Feature extraction requires a fresh output directory.


In [ ]:
from spDecon.config import SolverConfig
from spDecon.consensus import build_consensus
from spDecon.histology import fit_histology
initial = build_consensus(y, K, random_state=1,
    solver_config=SolverConfig(device=DEVICE, random_state=1, verbose=VERBOSE))
shared = dict(initial_proportions=initial.consensus_proportions,
    initial_profiles=initial.consensus_profiles, sections=sections,
    random_state=1, device=DEVICE, max_iter=MAX_ITER, verbose=VERBOSE)
he = fit_histology(y, q, xy, K, image_weight=None, spatial_weight=None, **shared)
rna = fit_histology(y, None, xy, K, spatial_weight=he.diagnostics["spatial_weight"], **shared)
fits = {"RNA_TV": rna.proportions, "RNA_HE_TV": he.proportions}
display(he.diagnostics)
display(rna.diagnostics)


## 5. Display components and save the fitted weights


In [ ]:
p = he.proportions
components = [f"C{k+1}" for k in range(K)]

fig, axes = plt.subplots(int(np.ceil(p.shape[1]/4)), 4,
                         figsize=(12, 3*int(np.ceil(p.shape[1]/4))), squeeze=False)
for k, ax in enumerate(axes.ravel()):
    if k < p.shape[1]:
        art = spatial(ax, xy, p[:,k], components[k], size=10)
    else:
        ax.axis("off")
fig.colorbar(art, ax=axes, label="Molecular contribution", shrink=.6)
save_figure(fig, "histology_components")

adata.obsm["spDecon_HE"] = he.proportions
adata.obsm["spDecon_RNA"] = rna.proportions
out = ROOT / "outputs/histology"
out.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(out / "deconvolved.h5ad")


## 6. Optional evaluation with region annotations

Set `REGION_COLUMN` to an annotation column to evaluate labelled spots. ARI compares dominant
components with region labels; boundary F1 compares cross-region edges on the same graph.
Component-to-region PCC is reported when the component and region counts agree. Matching and
scoring use the labelled spots, giving descriptive recovery within this section.


In [ ]:
from sklearn.metrics import adjusted_rand_score, f1_score
if REGION_COLUMN is not None:
    annotation = adata.obs[REGION_COLUMN]
    support = np.flatnonzero(annotation.notna().to_numpy())
    if not len(support):
        raise ValueError("The supplied region column has no labelled spots")
    codes, labels = pd.factorize(annotation.iloc[support], sort=True)
    truth = np.eye(len(labels))[codes]
    position = np.full(len(y), -1)
    position[support] = np.arange(len(support))
    left, right = position[he.extra["edges"]].T
    keep = (left >= 0) & (right >= 0)
    left, right = left[keep], right[keep]
    rows = []
    for arm, weights in fits.items():
        predicted = weights[support].argmax(axis=1)
        row = {"model": arm, "ARI": adjusted_rand_score(codes, predicted), "n": len(support)}
        if len(left):
            row["boundary_F1"] = f1_score(codes[left] != codes[right],
                predicted[left] != predicted[right], zero_division=0)
        if len(labels) == K:
            matched, _ = align(weights[support], truth)
            row["mean_region_PCC"] = np.diag(correlation(matched, truth)).mean()
        rows.append(row)
    display(save_table(pd.DataFrame(rows), "histology_region_scores"))
else:
    print("No region labels supplied; fitted component maps are available above.")
